# NTU ML2021 Whisper-small LoRA

This notebook reproduces the OpenTypeless Mandarin-English technical-lecture ASR experiment. Use a Google Colab GPU runtime. It downloads public, pinned code and the public `ky552/ML2021_ASR_ST` dataset; no API key is required.

The development run uses 6,000 training samples and evaluates only `dev`. Keep `RUN_FINAL_TEST` disabled until you have frozen every development choice.

In [ ]:
import re
import subprocess
import sys
from pathlib import Path

REPOSITORY = 'https://github.com/ryrenz/open-typeless-formac.git'
EXPERIMENT_REVISION = '44a60455983cd84da27cc624be4aade89fd930b3'
EXPERIMENT_ROOT = Path('/content/open-typeless-formac')

if not re.fullmatch(r'[0-9a-f]{40}', EXPERIMENT_REVISION):
    raise ValueError('Set EXPERIMENT_REVISION to the exact 40-character Git commit that contains this notebook.')

subprocess.run(['git', 'clone', REPOSITORY, str(EXPERIMENT_ROOT)], check=True)
subprocess.run(['git', '-C', str(EXPERIMENT_ROOT), 'checkout', '--detach', EXPERIMENT_REVISION], check=True)
EXPERIMENT_DIR = EXPERIMENT_ROOT / 'training' / 'ntu_ml2021'
if not EXPERIMENT_DIR.is_dir():
    raise RuntimeError('The selected commit does not contain training/ntu_ml2021.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--require-hashes', '-r', 'requirements.txt'], cwd=EXPERIMENT_DIR, check=True)
sys.path.insert(0, str(EXPERIMENT_DIR))
print(subprocess.check_output(['git', '-C', str(EXPERIMENT_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
subprocess.run([sys.executable, '-m', 'ntuasr.prepare_dataset', '--output-dir', 'artifacts/manifest'], cwd=EXPERIMENT_DIR, check=True)

In [ ]:
RUN_FULL_TRAINING = False
RUN_FINAL_TEST = False
MAX_TRAIN_SAMPLES = None if RUN_FULL_TRAINING else 6000
MAX_EVAL_SAMPLES = 512
OUTPUT_DIR = 'artifacts/whisper-small-lora-full' if RUN_FULL_TRAINING else 'artifacts/whisper-small-lora'
print({'max_train_samples': MAX_TRAIN_SAMPLES, 'max_eval_samples': MAX_EVAL_SAMPLES, 'output_dir': OUTPUT_DIR, 'run_final_test': RUN_FINAL_TEST})

In [ ]:
command = [
    sys.executable, '-m', 'ntuasr.train',
    '--output-dir', OUTPUT_DIR,
    '--max-eval-samples', str(MAX_EVAL_SAMPLES),
    '--num-train-epochs', '1',
]
if MAX_TRAIN_SAMPLES is not None:
    command.extend(['--max-train-samples', str(MAX_TRAIN_SAMPLES)])
subprocess.run(command, cwd=EXPERIMENT_DIR, check=True)

In [ ]:
DEVELOPMENT_EVALUATION_SAMPLES = 512

for adapter, output_name in [
    (None, 'base-dev-prefix-512.json'),
    (f'{OUTPUT_DIR}/adapter', 'fine-tuned-dev-prefix-512.json'),
]:
    command = [
        sys.executable, '-m', 'ntuasr.evaluate',
        '--split', 'dev',
        '--max-samples', str(DEVELOPMENT_EVALUATION_SAMPLES),
        '--output', f'artifacts/evaluation/{output_name}',
    ]
    if adapter:
        command.extend(['--adapter', adapter])
    subprocess.run(command, cwd=EXPERIMENT_DIR, check=True)

In [ ]:
if RUN_FINAL_TEST:
    for adapter, output_name in [
        (None, 'base-test-full.json'),
        (f'{OUTPUT_DIR}/adapter', 'fine-tuned-test-full.json'),
    ]:
        command = [
            sys.executable, '-m', 'ntuasr.evaluate',
            '--split', 'test',
            '--output', f'artifacts/evaluation/{output_name}',
        ]
        if adapter:
            command.extend(['--adapter', adapter])
        subprocess.run(command, cwd=EXPERIMENT_DIR, check=True)
else:
    print('Final test remains locked. Set RUN_FINAL_TEST = True only after freezing the development protocol.')

## Optional GGML export

Run this only after evaluation. It checks out fixed converter revisions, writes a temporary WAV from the official `dev` split, and verifies both exported GGML files with `whisper-cli`. Copy the resulting model and aggregate evaluation report to persistent storage before the Colab runtime ends.

In [ ]:
OPENAI_WHISPER_REVISION = '31243bad24cc746f07d4c8bfdd2d974872cb1803'
WHISPER_CPP_REVISION = '23ee03506a91ac3d3f0071b40e66a430eebdfa1d'
OPENAI_WHISPER_DIR = Path('/content/openai-whisper')
WHISPER_CPP_DIR = Path('/content/whisper.cpp')

def clone_at_revision(repository, target, revision):
    subprocess.run(['git', 'clone', repository, str(target)], check=True)
    subprocess.run(['git', '-C', str(target), 'checkout', '--detach', revision], check=True)

clone_at_revision('https://github.com/openai/whisper.git', OPENAI_WHISPER_DIR, OPENAI_WHISPER_REVISION)
clone_at_revision('https://github.com/ggml-org/whisper.cpp.git', WHISPER_CPP_DIR, WHISPER_CPP_REVISION)
subprocess.run(['cmake', '-S', str(WHISPER_CPP_DIR), '-B', str(WHISPER_CPP_DIR / 'build')], check=True)
subprocess.run(['cmake', '--build', str(WHISPER_CPP_DIR / 'build'), '--target', 'whisper-cli', 'quantize', '-j', '2'], check=True)

In [ ]:
import soundfile as sf
from datasets import Audio, load_dataset
from ntuasr.constants import DATASET_ID, DATASET_REVISION

smoke_dataset = load_dataset(DATASET_ID, revision=DATASET_REVISION, split='dev').cast_column('audio', Audio(sampling_rate=16_000))
smoke_audio = smoke_dataset[0]['audio']
SMOKE_AUDIO_PATH = Path('/content/ntu-ml2021-dev-smoke.wav')
sf.write(SMOKE_AUDIO_PATH, smoke_audio['array'], smoke_audio['sampling_rate'])

subprocess.run([
    'bash', 'scripts/export_ggml.sh',
    f'{OUTPUT_DIR}/merged',
    'artifacts/ggml',
    str(WHISPER_CPP_DIR),
    str(OPENAI_WHISPER_DIR),
    str(WHISPER_CPP_DIR / 'build/bin/whisper-cli'),
    str(SMOKE_AUDIO_PATH),
], cwd=EXPERIMENT_DIR, check=True)